In [7]:
# CELL 1: COMPLETE SETUP
!pip install pandas scikit-learn matplotlib sympy --target=/venv/main/lib/python3.12/site-packages

import re, time, random, pickle, urllib.request
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
import sympy as sp
from sympy import (symbols, series, sin, cos, exp, log, tan, sqrt,
                   atan, asin, acos, sinh, cosh, tanh, Rational, expand, zoo, nan, oo, I, Symbol)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0)}")
x = Symbol('x')

# --- Tokenizer ---
class MathTokenizer:
    PAD_TOKEN='<PAD>'; SOS_TOKEN='<SOS>'; EOS_TOKEN='<EOS>'; UNK_TOKEN='<UNK>'
    def __init__(self):
        self.token2idx={}; self.idx2token={}; self.vocab_size=0
        for i,t in enumerate([self.PAD_TOKEN,self.SOS_TOKEN,self.EOS_TOKEN,self.UNK_TOKEN]):
            self.token2idx[t]=i; self.idx2token[i]=t
        self.vocab_size=4
    def tokenize(self, s):
        s=s.replace(" ","")
        raw=re.findall(r'(sin|cos|tan|exp|log|sqrt|sinh|cosh|tanh|asin|acos|atan|\*\*|\d+|\+|\-|\*|/|\(|\)|x)',s)
        tokens=[]
        for t in raw:
            if t.isdigit() and len(t)>1:
                for d in t: tokens.append(d)
            else: tokens.append(t)
        return tokens
    def build_vocab(self, exprs=None):
        for t in ['0','1','2','3','4','5','6','7','8','9','+','-','*','/','**','(',')','x',
                   'sin','cos','tan','exp','log','sqrt','sinh','cosh','tanh','asin','acos','atan']:
            if t not in self.token2idx: self.token2idx[t]=self.vocab_size; self.idx2token[self.vocab_size]=t; self.vocab_size+=1
        if exprs:
            for e in exprs:
                for t in self.tokenize(e):
                    if t not in self.token2idx: self.token2idx[t]=self.vocab_size; self.idx2token[self.vocab_size]=t; self.vocab_size+=1
        print(f"Vocabulary: {self.vocab_size} tokens"); return self
    def encode(self, s, add_sos=False, add_eos=False):
        ids=[]
        if add_sos: ids.append(self.token2idx[self.SOS_TOKEN])
        for t in self.tokenize(s): ids.append(self.token2idx.get(t, self.token2idx[self.UNK_TOKEN]))
        if add_eos: ids.append(self.token2idx[self.EOS_TOKEN])
        return ids
    def decode(self, ids):
        tokens=[self.idx2token.get(i,self.UNK_TOKEN) for i in ids if self.idx2token.get(i,'') not in [self.PAD_TOKEN,self.SOS_TOKEN,self.EOS_TOKEN]]
        r=''; i=0
        while i<len(tokens):
            t=tokens[i]
            if t.isdigit():
                n=t
                while i+1<len(tokens) and tokens[i+1].isdigit(): i+=1; n+=tokens[i]
                r+=n
            elif t in ['+','-']: r+=' '+t+' '
            else: r+=t
            i+=1
        return r.strip()
    def pad_sequence(self, ids, ml):
        return ids[:ml] if len(ids)>=ml else ids+[self.token2idx[self.PAD_TOKEN]]*(ml-len(ids))

# --- Download data ---
print("Downloading 60K dataset...")
urllib.request.urlretrieve("https://raw.githubusercontent.com/hbprosper/symbolic-ai/main/data/seq2seq_data_60000.txt", "data.txt")
pairs=[]
for line in open("data.txt"):
    parts=line.strip().split('\t')
    if len(parts)==2: pairs.append({'function':parts[0],'taylor_expansion':parts[1]})
df=pd.DataFrame(pairs).drop_duplicates(subset=['function']).reset_index(drop=True)
print(f"Loaded: {len(df)} pairs")

tokenizer=MathTokenizer()
tokenizer.build_vocab(df['function'].tolist()+df['taylor_expansion'].tolist())
VS=tokenizer.vocab_size

fl=[len(tokenizer.tokenize(f)) for f in df['function']]
tl=[len(tokenizer.tokenize(t)) for t in df['taylor_expansion']]
MAX_IN=int(np.percentile(fl,95))+5
MAX_OUT=min(int(np.percentile(tl,95))+5, 120)

df=df[(df['function'].apply(lambda f:len(tokenizer.tokenize(f))<=MAX_IN-3))&
      (df['taylor_expansion'].apply(lambda t:len(tokenizer.tokenize(t))<=MAX_OUT-3))].reset_index(drop=True)
print(f"After filtering: {len(df)} | MAX_IN={MAX_IN} MAX_OUT={MAX_OUT} VOCAB={VS}")

train_df,temp=train_test_split(df,test_size=0.15,random_state=SEED)
val_df,test_df=train_test_split(temp,test_size=0.33,random_state=SEED)
print(f"Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}")

class TaylorDataset(Dataset):
    def __init__(self,df,tok,mi,mo):
        self.data=df.reset_index(drop=True);self.tok=tok;self.mi=mi;self.mo=mo
    def __len__(self):return len(self.data)
    def __getitem__(self,i):
        r=self.data.iloc[i]
        return{'src':torch.tensor(self.tok.pad_sequence(self.tok.encode(r['function']),self.mi),dtype=torch.long),
               'tgt':torch.tensor(self.tok.pad_sequence(self.tok.encode(r['taylor_expansion'],add_sos=True,add_eos=True),self.mo),dtype=torch.long)}

BS=128
train_loader=DataLoader(TaylorDataset(train_df,tokenizer,MAX_IN,MAX_OUT),batch_size=BS,shuffle=True,num_workers=4,pin_memory=True)
val_loader=DataLoader(TaylorDataset(val_df,tokenizer,MAX_IN,MAX_OUT),batch_size=BS,num_workers=4,pin_memory=True)
test_loader=DataLoader(TaylorDataset(test_df,tokenizer,MAX_IN,MAX_OUT),batch_size=BS,num_workers=4,pin_memory=True)

# --- Model definitions ---
EMBED=128;HIDDEN=256;NL=2;DROP=0.3;LR=1e-3

class LSTMEncoder(nn.Module):
    def __init__(self,vs,ed,hd,nl,do):
        super().__init__()
        self.emb=nn.Embedding(vs,ed,padding_idx=0);self.lstm=nn.LSTM(ed,hd,nl,batch_first=True,dropout=do if nl>1 else 0,bidirectional=True)
        self.fc_h=nn.Linear(hd*2,hd);self.fc_c=nn.Linear(hd*2,hd);self.drop=nn.Dropout(do)
    def forward(self,src):
        out,(h,c)=self.lstm(self.drop(self.emb(src)))
        h=torch.tanh(self.fc_h(torch.cat([h[-2],h[-1]],dim=1))).unsqueeze(0)
        c=torch.tanh(self.fc_c(torch.cat([c[-2],c[-1]],dim=1))).unsqueeze(0)
        return out,h,c

class Attention(nn.Module):
    def __init__(self,hd):
        super().__init__();self.attn=nn.Linear(hd*3,hd);self.v=nn.Linear(hd,1,bias=False)
    def forward(self,hidden,enc_out):
        h=hidden.squeeze(0).unsqueeze(1).repeat(1,enc_out.shape[1],1)
        return torch.softmax(self.v(torch.tanh(self.attn(torch.cat([h,enc_out],dim=2)))).squeeze(2),dim=1)

class LSTMDecoder(nn.Module):
    def __init__(self,vs,ed,hd,nl,do):
        super().__init__()
        self.emb=nn.Embedding(vs,ed,padding_idx=0);self.attn=Attention(hd)
        self.lstm=nn.LSTM(ed+hd*2,hd,1,batch_first=True);self.fc=nn.Linear(hd*3+ed,vs);self.drop=nn.Dropout(do)
    def forward(self,tok,h,c,enc_out):
        emb=self.drop(self.emb(tok));aw=self.attn(h,enc_out).unsqueeze(1);ctx=torch.bmm(aw,enc_out)
        out,(h,c)=self.lstm(torch.cat([emb,ctx],dim=2),(h,c))
        return self.fc(torch.cat([out,ctx,emb],dim=2)).squeeze(1),h,c

class Seq2SeqLSTM(nn.Module):
    def __init__(self,vs,ed,hd,nl,do):
        super().__init__();self.enc=LSTMEncoder(vs,ed,hd,nl,do);self.dec=LSTMDecoder(vs,ed,hd,nl,do);self.vs=vs
    def forward(self,src,tgt,tf=0.5):
        enc_out,h,c=self.enc(src);B,T=tgt.shape[0],tgt.shape[1]
        outputs=torch.zeros(B,T,self.vs).to(src.device);inp=tgt[:,0:1]
        for t in range(1,T):
            pred,h,c=self.dec(inp,h,c,enc_out);outputs[:,t]=pred
            inp=tgt[:,t:t+1] if random.random()<tf else pred.argmax(dim=1,keepdim=True)
        return outputs

class PositionalEncoding(nn.Module):
    def __init__(self,d,ml=500,do=0.1):
        super().__init__();self.drop=nn.Dropout(do)
        pe=torch.zeros(ml,d);pos=torch.arange(0,ml,dtype=torch.float).unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x):return self.drop(x+self.pe[:,:x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self,vs,dm,nh,nl,df,do):
        super().__init__();self.dm=dm
        self.src_emb=nn.Embedding(vs,dm,padding_idx=0);self.tgt_emb=nn.Embedding(vs,dm,padding_idx=0)
        self.pos=PositionalEncoding(dm,500,do)
        self.transformer=nn.Transformer(d_model=dm,nhead=nh,num_encoder_layers=nl,num_decoder_layers=nl,dim_feedforward=df,dropout=do,batch_first=True)
        self.fc=nn.Linear(dm,vs);self.drop=nn.Dropout(do)
    def make_mask(self,sz):return torch.triu(torch.ones(sz,sz,device=device),diagonal=1).bool()
    def forward(self,src,tgt):
        tm=self.make_mask(tgt.shape[1]);sp=(src==0);tp=(tgt==0)
        s=self.pos(self.drop(self.src_emb(src)*np.sqrt(self.dm)))
        t=self.pos(self.drop(self.tgt_emb(tgt)*np.sqrt(self.dm)))
        return self.fc(self.transformer(s,t,tgt_mask=tm,src_key_padding_mask=sp,tgt_key_padding_mask=tp,memory_key_padding_mask=sp))

# Load LSTM (pretrained from H100 — download weights from GitHub)
lstm_model=Seq2SeqLSTM(VS,EMBED,HIDDEN,NL,DROP).to(device)

# Load Transformer v1 (pretrained)
t_model=Seq2SeqTransformer(VS,256,8,4,1024,DROP).to(device)

# Greedy decoders
def greedy_decode_lstm(mdl,src,tok,max_len):
    mdl.eval()
    with torch.no_grad():
        src=src.unsqueeze(0).to(device);enc_out,h,c=mdl.enc(src)
        inp=torch.tensor([[tok.token2idx[tok.SOS_TOKEN]]]).to(device);tokens=[]
        for _ in range(max_len):
            pred,h,c=mdl.dec(inp,h,c,enc_out);next_t=pred.argmax(dim=1).item()
            if next_t==tok.token2idx[tok.EOS_TOKEN]:break
            tokens.append(next_t);inp=torch.tensor([[next_t]]).to(device)
    return tokens

def greedy_decode_transformer(mdl,src,tok,max_len):
    mdl.eval()
    with torch.no_grad():
        src=src.unsqueeze(0).to(device);sos=tok.token2idx[tok.SOS_TOKEN];eos=tok.token2idx[tok.EOS_TOKEN]
        tgt_ids=[sos]
        for _ in range(max_len):
            tgt_t=torch.tensor([tgt_ids]).to(device);out=mdl(src,tgt_t)
            next_t=out[0,-1].argmax().item()
            if next_t==eos:break
            tgt_ids.append(next_t)
    return tgt_ids[1:]

print("✅ All setup complete! Ready for beam search + Transformer v2 training.")

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached f

# ============================================================
# BEAM SEARCH DECODING
# ============================================================

In [8]:


def beam_search_lstm(model, src, tokenizer, max_len, beam_width=5):
    """Beam search for LSTM seq2seq"""
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        enc_out, h, c = model.enc(src)
        
        sos = tokenizer.token2idx[tokenizer.SOS_TOKEN]
        eos = tokenizer.token2idx[tokenizer.EOS_TOKEN]
        
        # Each beam: (score, token_ids, hidden, cell)
        beams = [(0.0, [sos], h, c)]
        completed = []
        
        for _ in range(max_len):
            candidates = []
            for score, seq, h_b, c_b in beams:
                if seq[-1] == eos:
                    completed.append((score, seq[1:]))  # Remove SOS
                    continue
                
                inp = torch.tensor([[seq[-1]]]).to(device)
                pred, h_new, c_new = model.dec(inp, h_b, c_b, enc_out)
                log_probs = torch.log_softmax(pred, dim=-1)
                
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for i in range(beam_width):
                    new_score = score + topk_probs[0, i].item()
                    new_seq = seq + [topk_ids[0, i].item()]
                    candidates.append((new_score, new_seq, h_new, c_new))
            
            if not candidates:
                break
            
            # Keep top-k beams
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]
        
        # Add remaining beams to completed
        for score, seq, _, _ in beams:
            completed.append((score / max(len(seq)-1, 1), seq[1:]))
        
        if not completed:
            return []
        
        # Return best (length-normalized)
        completed.sort(key=lambda x: x[0], reverse=True)
        result = completed[0][1]
        # Remove EOS if present
        if result and result[-1] == eos:
            result = result[:-1]
        return result


def beam_search_transformer(model, src, tokenizer, max_len, beam_width=5):
    """Beam search for Transformer seq2seq"""
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        sos = tokenizer.token2idx[tokenizer.SOS_TOKEN]
        eos = tokenizer.token2idx[tokenizer.EOS_TOKEN]
        
        # Each beam: (score, token_ids)
        beams = [(0.0, [sos])]
        completed = []
        
        for _ in range(max_len):
            candidates = []
            for score, seq in beams:
                if seq[-1] == eos:
                    completed.append((score / max(len(seq)-1, 1), seq[1:]))
                    continue
                
                tgt = torch.tensor([seq]).to(device)
                out = model(src, tgt)
                log_probs = torch.log_softmax(out[0, -1], dim=-1)
                
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for i in range(beam_width):
                    new_score = score + topk_probs[i].item()
                    new_seq = seq + [topk_ids[i].item()]
                    candidates.append((new_score, new_seq))
            
            if not candidates:
                break
            
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]
        
        for score, seq in beams:
            completed.append((score / max(len(seq)-1, 1), seq[1:]))
        
        if not completed:
            return []
        
        completed.sort(key=lambda x: x[0], reverse=True)
        result = completed[0][1]
        if result and result[-1] == eos:
            result = result[:-1]
        return result

print("✅ Beam search decoders ready!")

✅ Beam search decoders ready!


# Clear memory first

In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Free: 50.1 GB


In [ ]:
import gc
try:
    del t_model_v2
except:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Smaller batch loader for TF decay epochs
BS_SMALL = 32
train_loader_small = DataLoader(TaylorDataset(train_df,tokenizer,MAX_IN,MAX_OUT),batch_size=BS_SMALL,shuffle=True,num_workers=4,pin_memory=True)
val_loader_small = DataLoader(TaylorDataset(val_df,tokenizer,MAX_IN,MAX_OUT),batch_size=BS_SMALL,num_workers=4,pin_memory=True)

# Reinitialize
DM=256; NH=8; NL_T=4; DF=1024
t_model_v2 = Seq2SeqTransformerV2(VS, DM, NH, NL_T, DF, DROP).to(device)
print(f"Transformer v2 params: {sum(p.numel() for p in t_model_v2.parameters()):,}")

opt_v2 = torch.optim.Adam(t_model_v2.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9)
crit_v2 = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS_V2 = 50
warmup_steps = len(train_loader) * 3
total_steps = len(train_loader) * EPOCHS_V2
step = 0

def lr_lambda(s):
    if s < warmup_steps: return s / warmup_steps
    progress = (s - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))
sched_v2 = torch.optim.lr_scheduler.LambdaLR(opt_v2, lr_lambda)

tv2_hist = {'train': [], 'val': []}
best_vt2 = float('inf')

for epoch in range(EPOCHS_V2):
    t_model_v2.train(); tl = 0
    
    # First 10 epochs: full TF with big batches (fast)
    # After: TF decay with small batches (fits in memory)
    if epoch < 10:
        tf_ratio = 1.0
        loader = train_loader
    else:
        tf_ratio = max(0.3, 1.0 - (epoch - 10) * 0.025)
        loader = train_loader_small
    
    for batch in loader:
        src, tgt = batch['src'].to(device), batch['tgt'].to(device)
        opt_v2.zero_grad()
        out = t_model_v2(src, tgt[:, :-1], teacher_forcing_ratio=tf_ratio)
        out = out.contiguous().view(-1, VS)
        loss = crit_v2(out, tgt[:, 1:].contiguous().view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(t_model_v2.parameters(), 1.0)
        opt_v2.step(); sched_v2.step()
        tl += loss.item(); step += 1
        
        # Clear cache after each batch during TF decay
        if tf_ratio < 1.0:
            torch.cuda.empty_cache()
    
    tl /= len(loader)
    
    t_model_v2.eval(); vl = 0
    with torch.no_grad():
        for batch in val_loader:
            src, tgt = batch['src'].to(device), batch['tgt'].to(device)
            out = t_model_v2(src, tgt[:, :-1], teacher_forcing_ratio=1.0)
            out = out.contiguous().view(-1, VS)
            vl += crit_v2(out, tgt[:, 1:].contiguous().view(-1)).item()
    vl /= len(val_loader)
    tv2_hist['train'].append(tl); tv2_hist['val'].append(vl)
    
    star = ''
    if vl < best_vt2:
        best_vt2 = vl
        torch.save(t_model_v2.state_dict(), 'best_transformer_v2.pth')
        star = ' ★'
    
    clr = opt_v2.param_groups[0]['lr']
    print(f"Ep {epoch+1:2d}/{EPOCHS_V2} | Train: {tl:.4f} | Val: {vl:.4f} | LR: {clr:.6f} | TF: {tf_ratio:.2f}{star}")

print(f"\nBest Transformer v2 val loss: {best_vt2:.4f}")

Free: 50.1 GB
Transformer v2 params: 7,399,970
Ep  1/50 | Train: 2.2857 | Val: 1.5661 | LR: 0.000167 | TF: 1.00 ★
Ep  2/50 | Train: 1.4335 | Val: 1.1915 | LR: 0.000333 | TF: 1.00 ★
Ep  3/50 | Train: 1.2100 | Val: 1.0313 | LR: 0.000500 | TF: 1.00 ★
Ep  4/50 | Train: 1.0649 | Val: 0.9155 | LR: 0.000499 | TF: 1.00 ★
Ep  5/50 | Train: 0.9808 | Val: 0.8649 | LR: 0.000498 | TF: 1.00 ★
Ep  6/50 | Train: 0.9262 | Val: 0.8078 | LR: 0.000495 | TF: 1.00 ★
Ep  7/50 | Train: 0.8855 | Val: 0.7758 | LR: 0.000491 | TF: 1.00 ★
Ep  8/50 | Train: 0.8524 | Val: 0.7555 | LR: 0.000486 | TF: 1.00 ★
Ep  9/50 | Train: 0.8274 | Val: 0.7220 | LR: 0.000480 | TF: 1.00 ★
Ep 10/50 | Train: 0.8068 | Val: 0.7073 | LR: 0.000473 | TF: 1.00 ★
Ep 11/50 | Train: 0.8246 | Val: 0.6998 | LR: 0.000435 | TF: 1.00 ★


# ============================================================
# EVALUATION: Greedy vs Beam Search, All Models
# ============================================================

In [ ]:

print("="*60)
print("FULL EVALUATION: GREEDY vs BEAM SEARCH")
print("="*60)

def evaluate_model_full(model, greedy_fn, beam_fn, name, test_df, tok, max_in, max_out, beam_width=5):
    """Evaluate with both greedy and beam search"""
    print(f"\n--- {name} ---")
    
    greedy_results = {'exact': 0, 'tok_correct': 0, 'tok_total': 0, 'bleus': []}
    beam_results = {'exact': 0, 'tok_correct': 0, 'tok_total': 0, 'bleus': []}
    details = []
    
    for idx in range(len(test_df)):
        row = test_df.iloc[idx]
        src = tok.pad_sequence(tok.encode(row['function']), max_in)
        src_t = torch.tensor(src, dtype=torch.long)
        true_str = row['taylor_expansion']
        true_idx = tok.encode(row['taylor_expansion'])
        
        # Greedy decode
        g_idx = greedy_fn(model, src_t, tok, max_out)
        g_str = tok.decode(g_idx)
        
        # Beam search decode
        b_idx = beam_fn(model, src_t, tok, max_out, beam_width)
        b_str = tok.decode(b_idx)
        
        # Evaluate greedy
        g_exact = (g_str.replace(" ","") == true_str.replace(" ",""))
        if g_exact: greedy_results['exact'] += 1
        mn = min(len(g_idx), len(true_idx))
        gc = sum(1 for i in range(mn) if g_idx[i] == true_idx[i])
        greedy_results['tok_correct'] += gc
        greedy_results['tok_total'] += max(len(g_idx), len(true_idx))
        g_tok = tok.tokenize(g_str) if g_str else []
        t_tok = tok.tokenize(true_str)
        if g_tok:
            pc = Counter(g_tok); tc = Counter(t_tok)
            greedy_results['bleus'].append(sum(min(pc[t],tc[t]) for t in pc)/len(g_tok))
        else:
            greedy_results['bleus'].append(0.0)
        
        # Evaluate beam
        b_exact = (b_str.replace(" ","") == true_str.replace(" ",""))
        if b_exact: beam_results['exact'] += 1
        mn = min(len(b_idx), len(true_idx))
        bc = sum(1 for i in range(mn) if b_idx[i] == true_idx[i])
        beam_results['tok_correct'] += bc
        beam_results['tok_total'] += max(len(b_idx), len(true_idx))
        b_tok = tok.tokenize(b_str) if b_str else []
        if b_tok:
            pc = Counter(b_tok); tc = Counter(t_tok)
            beam_results['bleus'].append(sum(min(pc[t],tc[t]) for t in pc)/len(b_tok))
        else:
            beam_results['bleus'].append(0.0)
        
        details.append({
            'function': row['function'], 'true': true_str,
            'greedy_pred': g_str, 'beam_pred': b_str,
            'greedy_exact': g_exact, 'beam_exact': b_exact
        })
    
    n = len(test_df)
    g = {
        'exact': greedy_results['exact']/n*100,
        'tok': greedy_results['tok_correct']/max(greedy_results['tok_total'],1)*100,
        'bleu': np.mean(greedy_results['bleus'])*100
    }
    b = {
        'exact': beam_results['exact']/n*100,
        'tok': beam_results['tok_correct']/max(beam_results['tok_total'],1)*100,
        'bleu': np.mean(beam_results['bleus'])*100
    }
    
    print(f"  {'Metric':<20s} {'Greedy':>10s} {'Beam(k={beam_width})':>12s} {'Improvement':>12s}")
    print(f"  {'-'*56}")
    print(f"  {'Exact Match %':<20s} {g['exact']:>9.2f}% {b['exact']:>11.2f}% {b['exact']-g['exact']:>+11.2f}%")
    print(f"  {'Token Accuracy %':<20s} {g['tok']:>9.2f}% {b['tok']:>11.2f}% {b['tok']-g['tok']:>+11.2f}%")
    print(f"  {'BLEU-1 %':<20s} {g['bleu']:>9.2f}% {b['bleu']:>11.2f}% {b['bleu']-g['bleu']:>+11.2f}%")
    
    # Show beam search improvements
    ddf = pd.DataFrame(details)
    beam_fixed = ddf[(ddf['beam_exact']==True) & (ddf['greedy_exact']==False)]
    if len(beam_fixed) > 0:
        print(f"\n  Beam search FIXED {len(beam_fixed)} predictions that greedy got wrong:")
        for _, r in beam_fixed.head(5).iterrows():
            print(f"    {r['function'][:40]}")
            print(f"      Greedy: {r['greedy_pred'][:60]}")
            print(f"      Beam:   {r['beam_pred'][:60]} ✓")
    
    return {'greedy': g, 'beam': b, 'details': ddf, 'beam_bleus': beam_results['bleus']}

# Load best models
lstm_model.load_state_dict(torch.load('best_lstm_h100.pth', map_location=device))
t_model_v2.load_state_dict(torch.load('best_transformer_v2.pth', map_location=device))

# Also load original transformer for comparison
t_model.load_state_dict(torch.load('best_transformer_h100.pth', map_location=device))

# Evaluate all
lstm_res = evaluate_model_full(lstm_model, greedy_decode_lstm, beam_search_lstm,
                               "LSTM+Attention", test_df, tokenizer, MAX_IN, MAX_OUT, beam_width=5)

trans_v1_res = evaluate_model_full(t_model, greedy_decode_transformer, beam_search_transformer,
                                   "Transformer v1 (no TF decay)", test_df, tokenizer, MAX_IN, MAX_OUT, beam_width=5)

trans_v2_res = evaluate_model_full(t_model_v2, greedy_decode_transformer, beam_search_transformer,
                                   "Transformer v2 (with TF decay)", test_df, tokenizer, MAX_IN, MAX_OUT, beam_width=5)

# ============================================================
# FINAL COMPARISON TABLE
# ============================================================
print("\n" + "="*80)
print("FINAL MODEL COMPARISON")
print("="*80)
print(f"{'Model':<30s} {'Decode':<10s} {'Exact%':>8s} {'TokAcc%':>9s} {'BLEU%':>8s}")
print("-"*67)
print(f"{'Prosper Baseline':<30s} {'greedy':<10s} {'N/A':>8s} {'N/A':>9s} {'N/A':>8s}")
print(f"{'LSTM+Attention':<30s} {'greedy':<10s} {lstm_res['greedy']['exact']:>7.2f}% {lstm_res['greedy']['tok']:>8.2f}% {lstm_res['greedy']['bleu']:>7.2f}%")
print(f"{'LSTM+Attention':<30s} {'beam-5':<10s} {lstm_res['beam']['exact']:>7.2f}% {lstm_res['beam']['tok']:>8.2f}% {lstm_res['beam']['bleu']:>7.2f}%")
print(f"{'Transformer v1':<30s} {'greedy':<10s} {trans_v1_res['greedy']['exact']:>7.2f}% {trans_v1_res['greedy']['tok']:>8.2f}% {trans_v1_res['greedy']['bleu']:>7.2f}%")
print(f"{'Transformer v1':<30s} {'beam-5':<10s} {trans_v1_res['beam']['exact']:>7.2f}% {trans_v1_res['beam']['tok']:>8.2f}% {trans_v1_res['beam']['bleu']:>7.2f}%")
print(f"{'Transformer v2 (TF decay)':<30s} {'greedy':<10s} {trans_v2_res['greedy']['exact']:>7.2f}% {trans_v2_res['greedy']['tok']:>8.2f}% {trans_v2_res['greedy']['bleu']:>7.2f}%")
print(f"{'Transformer v2 (TF decay)':<30s} {'beam-5':<10s} {trans_v2_res['beam']['exact']:>7.2f}% {trans_v2_res['beam']['tok']:>8.2f}% {trans_v2_res['beam']['bleu']:>7.2f}%")

# Save
pickle.dump({
    'lstm': lstm_res, 'trans_v1': trans_v1_res, 'trans_v2': trans_v2_res,
    'tv2_hist': tv2_hist
}, open('final_results.pkl', 'wb'))

print("\n✅ All evaluations complete!")